# Second 1.0 — Stage T4: Reward v2 and GRPO v2 Optimization

This notebook runs **Stage T4 (GRPO v2)** on Google Colab using an **NVIDIA L4 or A100 GPU**.
It starts from the winning **`SFT-A1` (`r=32`)** adapter checkpoint from Stage T3.

### Why We Need Stage T4
- In Stage T3, `SFT-A1` fixed brevity (64.0 words, 99.3% in-band) and eliminated diary voice (0.0%).
- However, standard SFT cannot penalize hallucinated moves or wrong piece locations (`14.67%` hard-fail rate on `devset-1.0.jsonl`).
- Stage T4 uses **Reward v2** (graded penalties + multiplicative length scaling) and **GRPO v2** (`loss_type="dapo"`, 8 prompts per step) to push the hard-fail rate down to **≤ 5.0%** while keeping explanations detailed.

### Stage T4 Ablation Matrix (`TRAINING_PLAN.md` Section 8.6)

| Exp ID | Algorithm / Loss | Reward Formulation | Aggregation | Goal |
|---|---|---|---|---|
| `RL-B2` | GRPO (`loss_type="dapo"`, 8 prompts/step) | Reward v2 (Graded + Multiplicative Length) | `sum_then_normalize` | **Recommended primary run** (150–300 steps). |
| `RL-B3` | GRPO (`loss_type="dr_grpo"`, 8 prompts/step) | Reward v2 (Graded + Multiplicative Length) | `sum_then_normalize` | Compare constant normalization vs token normalization. |
| `RL-B4` | GRPO (`loss_type="dapo"`, 8 prompts/step) | Split 3-stream Reward v2 | `normalize_then_sum` (GDPO / MO-GRPO) | Independent advantage normalization per objective. |
| `RL-B1` | Rejection Sampling Fine-Tuning (RFT: best of 8 -> 1 epoch SFT) | Reward v2 (`>= 0.80` gate) | Offline | Optional offline RFT baseline. |

### Stage T4 Pass Gates (Evaluated on `eval/devset-1.0.jsonl`)
- **Dev Hard Fail Rate**: &le; 5.0% (down from 14.67%)
- **Dev In-Band Length**: &ge; 80.0%
- **Dev First-Person Tone**: &le; 2.0%
- **Dev Key Point Score**: &ge; 0.70
- **Dev Better Move Score**: &ge; 0.75
- **Evaluation Rule**: Never tune or train on `eval/testset-1.0.jsonl`. Evaluate strictly on `eval/devset-1.0.jsonl`.

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemError("No GPU detected. In Colab menu: Runtime -> Change runtime type -> L4 or A100 GPU.")

device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
bf16_supported = torch.cuda.is_bf16_supported()
print(f"GPU: {device_name} ({vram_gb:.1f} GB VRAM, bf16={bf16_supported})")


In [ ]:
# Step 2: Mount Google Drive and Locate Stage T3 Winner Checkpoint (SFT-A1)
import glob
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path("/content/drive/MyDrive/second")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted: {DRIVE_DIR}")
except Exception as exc:
    print(f"Drive mount notice: {exc}")
    DRIVE_DIR = Path("/content/second_local")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

SFT_A1_ROOT = DRIVE_DIR / "checkpoints" / "sft_v2_a1"
sft_candidates = [
    SFT_A1_ROOT / "final_adapter",
    SFT_A1_ROOT / "best_adapter",
    SFT_A1_ROOT,
]
step_ckpts = sorted(
    glob.glob(str(SFT_A1_ROOT / "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0,
)
if step_ckpts:
    sft_candidates.append(Path(step_ckpts[-1]))

SFT_ADAPTER_PATH = None
for cand in sft_candidates:
    if (cand / "adapter_config.json").is_file():
        SFT_ADAPTER_PATH = str(cand)
        break

if SFT_ADAPTER_PATH:
    print(f"Found Stage T3 SFT-A1 adapter: {SFT_ADAPTER_PATH}")
else:
    print(f"WARNING: SFT-A1 adapter not found at {SFT_A1_ROOT}. Verify Google Drive path.")

In [ ]:
# Step 3: Extract Stage T4 GRPO v2 Bundle
import shutil
import sys
import zipfile

WORK_DIR = Path("/content/Second")
WORK_DIR.mkdir(parents=True, exist_ok=True)

bundle_locations = [
    DRIVE_DIR / "data" / "phase4_grpo_v2_bundle.zip",
    Path("/content/phase4_grpo_v2_bundle.zip"),
    Path("data/processed/phase4_grpo_v2_bundle.zip"),
]

bundle_path = None
for loc in bundle_locations:
    if loc.is_file():
        bundle_path = loc
        break

if bundle_path and bundle_path.is_file():
    print(f"Extracting bundle from {bundle_path} into {WORK_DIR}...")
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall(WORK_DIR)
    os.chdir(str(WORK_DIR))
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    print(f"Bundle extracted. Working directory: {os.getcwd()}")
elif Path("training/grpo_data.py").is_file():
    print("Working in existing repository checkout.")
else:
    raise FileNotFoundError("phase4_grpo_v2_bundle.zip not found. Upload it to Google Drive or /content.")

In [ ]:
# Step 4: Install Dependencies
!pip uninstall -y -q torchao torchvision 2>/dev/null || true
!pip install -q "transformers>=4.49.0" "trl>=0.15.0" "accelerate>=1.2.0" "peft>=0.14.0" "datasets>=3.0.0" "bitsandbytes>=0.45.0" "chess" "pyyaml" "langdetect"
print("Dependencies installed successfully.")

In [ ]:
# Step 5: Select Stage T4 Experiment Configuration
# Options:
#   "RL-B2": GRPO with loss_type="dapo" + Reward v2 (Recommended primary run)
#   "RL-B3": GRPO with loss_type="dr_grpo" + Reward v2
#   "RL-B4": GRPO with 3-stream GDPO / MO-GRPO (normalize_then_sum)
EXPERIMENT_ID = "RL-B2"
MAX_STEPS_OVERRIDE = 150  # Set 150 for fast ablation or 300 for full training run
USE_CMDP_FALLBACK = False # Set True only if hard-fail > 5% or length drops > 15%

config_map = {
    "RL-B2": "training/configs/grpo_v2_b2_dapo.yaml",
    "RL-B3": "training/configs/grpo_v2_b3_drgrpo.yaml",
    "RL-B4": "training/configs/grpo_v2_b4_gdpo.yaml",
}

from training.grpo_data import load_grpo_config
config_file = config_map[EXPERIMENT_ID]
grpo_cfg = load_grpo_config(config_file)
print(f"Loaded Stage T4 config: {config_file}")
print(f"Experiment: {EXPERIMENT_ID} | Loss: {grpo_cfg.loss_type} | Reward: {grpo_cfg.reward_version} | Aggregation: {grpo_cfg.multi_objective_aggregation}")
print(f"Batch: per_device={grpo_cfg.per_device_train_batch_size} x grad_accum={grpo_cfg.gradient_accumulation_steps} (8 distinct prompts / 64 rollouts per step)")

In [ ]:
# Step 6: Load Base Qwen 3.5 2B and Attach Trainable SFT-A1 LoRA Adapter
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import peft.import_utils
import peft.tuners.lora.torchao

peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False

adapter_source = SFT_ADAPTER_PATH or grpo_cfg.sft_adapter_dir
if not os.path.isdir(adapter_source):
    raise FileNotFoundError(f"SFT-A1 adapter directory not found at {adapter_source}.")

tokenizer = AutoTokenizer.from_pretrained(adapter_source, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def load_policy_model(adapter_dir: str):
    gc.collect()
    torch.cuda.empty_cache()
    base_model = AutoModelForCausalLM.from_pretrained(
        grpo_cfg.base_model,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
        is_trainable=True,
    )
    peft_model.print_trainable_parameters()
    return peft_model

model = load_policy_model(adapter_source)

In [ ]:
# Step 7: Prepare RL v2 Dataset and Monitored Reward v2
from datasets import Dataset
from training.grpo_data import (
    MonitoredGRPOReward,
    build_multi_objective_reward_funcs,
    load_rl_prompt_records,
)

rl_records = load_rl_prompt_records(
    grpo_cfg.rl_prompts_path,
    max_prompts=5000,
    seed=42,
    shuffle=True,
)
rl_dataset = Dataset.from_list([{"prompt": r["prompt"], "id": r["id"]} for r in rl_records])
print(f"Loaded {len(rl_dataset)} difficulty-filtered RL v2 prompts from {grpo_cfg.rl_prompts_path}.")

reward_monitor = MonitoredGRPOReward(
    log_interval_batches=25,
    sample_size=20,
    seed=42,
    reward_version="v2",
    group_size=grpo_cfg.num_generations,
    use_cmdp_primal_dual=USE_CMDP_FALLBACK or grpo_cfg.use_cmdp_primal_dual,
)

# Sanity-check Reward v2 graded penalties and multiplicative length scaling
sample_p = rl_records[0]["prompt"]
good_text = (
    "This move is an inaccuracy that allows your opponent to seize central control. "
    "After this continuation, your pieces remain passive and lack coordination across the board. "
    "Instead, playing the recommended best move develops a piece and protects your king safety."
)
diary_text = "I was trying to attack, but I missed the tactic on my king."
sanity_scores = reward_monitor([sample_p, sample_p], [good_text, diary_text])
reward_monitor.call_count = 0
reward_monitor.current_window.clear()
print(f"Reward v2 sanity check -> good prose: {sanity_scores[0]:.4f} | diary voice fault (graded -0.35): {sanity_scores[1]:.4f}")

In [ ]:
# Step 8: Run Stage T4 GRPO v2 Training with Live Collapse & Entropy Telemetry
import inspect
import time
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

checkpoint_dir = Path(grpo_cfg.checkpoint_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
target_steps = MAX_STEPS_OVERRIDE or grpo_cfg.max_steps

class TelemetryPrintCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step > 0 and state.global_step % 25 == 0 and reward_monitor.step_summaries:
            s = reward_monitor.step_summaries[-1]
            print(
                f"\n[Step {state.global_step}] Reward v2: {s['mean_reward']:.4f} | "
                f"Hard Fail: {s['hard_fail_rate']:.1f}% | "
                f"Words: {s['avg_word_count']:.1f} (group_std={s['group_length_std']:.1f}) | "
                f"ZeroVar: {s['zero_var_group_pct']:.1f}% | "
                f"1stPerson: {s['first_person_pct']:.1f}% | "
                f"Motif/Best: {s['motif_mention_pct']:.1f}%"
            )
            if s.get("low_length_diversity"):
                print("  WARNING: group_length_std < 4.0 words. Watch for mode collapse.")

cfg_kwargs = {
    "output_dir": str(checkpoint_dir),
    "max_steps": target_steps,
    "per_device_train_batch_size": grpo_cfg.per_device_train_batch_size,
    "gradient_accumulation_steps": grpo_cfg.gradient_accumulation_steps,
    "num_generations": grpo_cfg.num_generations,
    "max_completion_length": grpo_cfg.max_completion_length,
    "temperature": grpo_cfg.temperature,
    "learning_rate": grpo_cfg.learning_rate,
    "beta": grpo_cfg.beta,
    "warmup_steps": grpo_cfg.warmup_steps,
    "logging_steps": grpo_cfg.logging_steps,
    "save_strategy": "steps",
    "save_steps": grpo_cfg.save_steps,
    "save_total_limit": grpo_cfg.save_total_limit,
    "bf16": True,
    "use_vllm": False,
    "report_to": "none",
}

sig = inspect.signature(GRPOConfig.__init__)
if "max_prompt_length" in sig.parameters:
    cfg_kwargs["max_prompt_length"] = grpo_cfg.max_prompt_length
if "loss_type" in sig.parameters:
    cfg_kwargs["loss_type"] = grpo_cfg.loss_type
if "multi_objective_aggregation" in sig.parameters:
    cfg_kwargs["multi_objective_aggregation"] = grpo_cfg.multi_objective_aggregation

grpo_args = GRPOConfig(**cfg_kwargs)

if grpo_cfg.multi_objective_aggregation == "normalize_then_sum":
    active_reward_funcs = build_multi_objective_reward_funcs(reward_monitor)
else:
    active_reward_funcs = [reward_monitor]

trainer_kwargs = {
    "model": model,
    "args": grpo_args,
    "train_dataset": rl_dataset,
    "callbacks": [TelemetryPrintCallback()],
}
t_sig = inspect.signature(GRPOTrainer.__init__)
if "reward_funcs" in t_sig.parameters:
    trainer_kwargs["reward_funcs"] = active_reward_funcs
else:
    trainer_kwargs["reward_function"] = reward_monitor
if "processing_class" in t_sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in t_sig.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

grpo_trainer = GRPOTrainer(**trainer_kwargs)
print(f"Starting Stage T4 GRPO v2 ({EXPERIMENT_ID}) for {target_steps} steps...")
t0 = time.time()
grpo_trainer.train()
elapsed_min = (time.time() - t0) / 60.0

final_adapter_dir = checkpoint_dir / "final_adapter"
model.save_pretrained(str(final_adapter_dir))
tokenizer.save_pretrained(str(final_adapter_dir))
print(f"GRPO v2 complete in {elapsed_min:.1f} min! Saved final adapter to {final_adapter_dir}")

In [ ]:
# Step 9: Generate Greedy Completions on Development Set (eval/devset-1.0.jsonl)
import json

model.eval()
model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
tokenizer.padding_side = "left"

devset_file = Path(grpo_cfg.devset_path)
with open(devset_file, "r", encoding="utf-8") as f:
    dev_items = [json.loads(line) for line in f if line.strip()]

print(f"Generating greedy completions on {len(dev_items)} devset items...")
output_records = []
batch_size = 8

for start_idx in range(0, len(dev_items), batch_size):
    batch = dev_items[start_idx : start_idx + batch_size]
    prompts = [
        f"<|im_start|>system\nYou are Second, a chess assistant. "
        f"You explain chess positions and moves clearly using the provided facts.<|im_end|>\n"
        f"<|im_start|>user\n{item['user_message']}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n"
        for item in batch
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    for item, out_ids, in_ids in zip(batch, outputs, inputs["input_ids"]):
        gen_ids = out_ids[len(in_ids):]
        comp = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        output_records.append({
            "id": item["id"],
            "completion": comp,
        })

dev_output_path = Path(f"eval/results/grpo_v2_{EXPERIMENT_ID.lower()}_dev_outputs.jsonl")
dev_output_path.parent.mkdir(parents=True, exist_ok=True)
with open(dev_output_path, "w", encoding="utf-8") as f:
    for r in output_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(output_records)} devset completions to {dev_output_path}")

In [ ]:
# Step 10: Run Stage T4 Gate Evaluation and Save Results to Google Drive
!python -m training.evaluate_grpo_v2 --devset eval/devset-1.0.jsonl --outputs eval/results/grpo_v2_{EXPERIMENT_ID.lower()}_dev_outputs.jsonl --experiment-id {EXPERIMENT_ID} --reports-dir eval/reports

drive_results_dir = DRIVE_DIR / "results" / EXPERIMENT_ID.lower()
drive_results_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(dev_output_path, drive_results_dir / dev_output_path.name)

report_path = Path(f"eval/reports/stage_t4_{EXPERIMENT_ID.lower()}_report.md")
if report_path.is_file():
    shutil.copy2(report_path, drive_results_dir / report_path.name)
    print(f"\nSaved evaluation report to Google Drive: {drive_results_dir / report_path.name}")

ledger_path = Path("eval/results/experiment_ledger.jsonl")
if ledger_path.is_file():
    shutil.copy2(ledger_path, drive_results_dir / ledger_path.name)

print(f"\nStage T4 GRPO v2 ({EXPERIMENT_ID}) Complete!")